## Concept 6 — Multi-Agent (Router Pattern)

### What is it?
No single agent is best at everything. Multi-Agent creates **specialized sub-agents**
— each tuned for one task type — and an **LLM router** that sends each query to the right specialist.

### Pipeline
```
Query
  → Router LLM: 'math' / 'docs' / 'weather'
  → Math Agent    (only has calculate tool)
  → Docs Agent    (only has search_docs tool)
  → Weather Agent (only has get_weather tool)
  → Final Answer
```

### Limitation (what Concept 7 fixes)
Still reactive — no upfront planning for multi-step complex tasks.
→ Concept 7 generates a full plan before executing any steps.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, calculate, search_docs, get_weather, TEST_QUERIES, run_queries
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, ToolMessage

llm = get_llm()
print('LLM ready — creating specialized agents...')

### Step 1 — Create Specialized Sub-Agents
Each agent has a focused system prompt and only the tools it needs.
Specialization reduces hallucination and wrong-tool selection.

In [ ]:
math_agent = create_agent(
    model=llm,
    tools=[calculate],
    system_prompt=(
        'You are a math specialist. '
        'Use the calculate tool for ALL arithmetic. '
        'Never compute in your head — always call the tool. '
        'Format answers concisely, e.g. "144 / 12 = 12.0"'
    ),
)

docs_agent = create_agent(
    model=llm,
    tools=[search_docs],
    system_prompt=(
        'You are a documentation specialist. '
        'Always use search_docs to find answers. '
        'Give clear, educational explanations based on what you find.'
    ),
)

weather_agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt=(
        'You are a weather assistant. '
        'Always use get_weather to look up weather. '
        'Report temperature and conditions clearly.'
    ),
)

print('Specialized agents created: math_agent, docs_agent, weather_agent')

### Step 2 — LLM Router
A separate LLM call classifies the query and routes to the right agent.

In [ ]:
def route(query: str) -> str:
    routing_prompt = (
        f'Classify this query into exactly one category:\n'
        f'- math     : arithmetic, calculation, numbers\n'
        f'- docs     : documentation, LangChain, agents, RAG, concepts\n'
        f'- weather  : weather, temperature, forecast\n'
        f'\nQuery: {query}\n'
        f'Answer with exactly one word: math, docs, or weather'
    )
    decision = llm.invoke(routing_prompt).content.strip().lower()
    # Normalise in case of extra words
    if 'math'    in decision: return 'math'
    if 'weather' in decision: return 'weather'
    return 'docs'

# Test the router on a few queries
test_routes = [
    'What is 144 divided by 12?',
    'Search docs for what is a LangChain agent?',
    'What is the weather in Hyderabad?',
]

print('Router decisions:')
for q in test_routes:
    print(f'  "{q}" → {route(q)}')

### Step 3 — Wire Router + Agents Together

In [ ]:
agent_registry = {
    'math':    math_agent,
    'docs':    docs_agent,
    'weather': weather_agent,
}

def multi_agent(query: str) -> str:
    decision = route(query)
    print(f'  [Router → {decision}_agent]')

    selected = agent_registry[decision]
    response = selected.invoke({'messages': [{'role': 'user', 'content': query}]})
    return response['messages'][-1].content

### Step 4 — Bonus: Keyword Router (No LLM Call)
A faster, cheaper alternative for well-defined categories.

In [ ]:
import re

MATH_KEYWORDS    = {'calculate', 'compute', 'multiply', 'divide', 'add', 'subtract', 'plus', 'minus', 'times', r'\d+\s*[*/+-]\s*\d+'}
WEATHER_KEYWORDS = {'weather', 'temperature', 'forecast', 'sunny', 'rainy', 'cloudy', 'degrees'}

def keyword_route(query: str) -> str:
    q = query.lower()
    if any(kw in q or re.search(kw, q) for kw in MATH_KEYWORDS):    return 'math'
    if any(kw in q for kw in WEATHER_KEYWORDS):                      return 'weather'
    return 'docs'

print('Keyword router decisions (no LLM call, instant):')
for q in TEST_QUERIES:
    print(f'  "{q[:50]}..." → {keyword_route(q)}')

### Test — Standard Queries

In [ ]:
run_queries(multi_agent, label='Concept 6 — Multi-Agent Router')